# 움직임 시험 7 — 영상 모델(Wan2.1 VACE 1.3B) + 참조 그림, 새 블로킹 두 가지: 각진 폭발(고정 카메라) / 각진 폭발 + 폭발 중심을 도는 카메라 (Kaggle T4, 학습 없음)
시험 3에서 VACE '참조 있음'이 장 사이 일관성·색면 단순함이 가장 좋았다(GPT 판정, docs/GPT-RESEARCH-MOTION-2026-10-26.md). GPT 지적: 둥근 꽃잎 블로킹을 따르면 둥근 폭발이 나온다.
- **ang**: 각진 불꽃 끝·한쪽으로 치우쳐 찢어지는 심을 넣은 블로킹(`fx_blocking.py --angular`), 고정 카메라.
- **orbit**: 같은 폭발 + 2D 시차 카메라(`--orbit 0.5`): 빌딩 3층이 다른 속도로 움직이고 가까운 빌딩이 폭발 앞을 지나감. 3D 없음.
- 참조 그림·글·설정은 시험 3과 같음(480×832, 30단계, guidance 5, 시드 1234). 글만 orbit은 '카메라가 폭발 둘레를 천천히 돎'으로.
- 설정: **GPU T4 x2**, **Internet On** → Save & Run All. 약 70–90분(시험 3: 한 판 약 30분).
- 끝나면 `motion7_result.zip`(나란히 영상 `motion7.mp4` + 장별 그림)을 Drive 공유 링크로 알려 준다.


In [ ]:
# 1) 패키지와 자료
import os, glob
COLAB = "COLAB_RELEASE_TAG" in os.environ or os.path.exists("/content")
OUT = "/content/out" if COLAB and not os.path.exists("/kaggle/working") else "/kaggle/working"
os.makedirs(OUT, exist_ok=True); print("결과 폴더", OUT)
!pip install -q "diffusers==0.35.1" "transformers==4.56.2" "accelerate>=1.0" ftfy imageio imageio-ffmpeg
os.system('curl -sSL -o /tmp/blk7.zip "https://drive.usercontent.google.com/download?id=1ZkTCeGVrhy1Alg898iwKDPjObEA5VHRI&export=download&confirm=t"')
os.system("mkdir -p /tmp/blk7 && cd /tmp/blk7 && unzip -q -o /tmp/blk7.zip")
for name in ("ang", "orbit"):
    print(name, len(glob.glob(f"/tmp/blk7/{name}/*.png")), "장")
!curl -sSL -o /tmp/ref.png "https://drive.usercontent.google.com/download?id=1pWgM5zWc6KK5-lBRmOs6zjovMZj-D-yG&export=download&confirm=t" && ls -la /tmp/ref.png


In [ ]:
# 2) Wan2.1 VACE 1.3B (diffusers). VAE는 fp32, 나머지는 bf16(T4는 에뮬레이션) — 안 되면 fp16
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
from diffusers import AutoencoderKLWan, WanVACEPipeline
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler
DT = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print("dtype", DT)
mid = "Wan-AI/Wan2.1-VACE-1.3B-diffusers"
vae = AutoencoderKLWan.from_pretrained(mid, subfolder="vae", torch_dtype=torch.float32)
pipe = WanVACEPipeline.from_pretrained(mid, vae=vae, torch_dtype=DT)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=3.0)
pipe.enable_model_cpu_offload()  # 글 인코더(큼)는 쓸 때만 GPU로
# 1회차(10-26) Kaggle T4: VAE(fp32)가 480×832×17장을 한 번에 처리하다 메모리 초과(F.normalize) → 타일·장 나눠 처리
pipe.vae.enable_tiling()
pipe.vae.enable_slicing()


In [ ]:
# 3) 새 블로킹 두 가지 × 참조 있음
import numpy as np, json
from PIL import Image
P = ("2D hand-drawn anime effects animation, sakuga, a fiery explosion over a night city: a white-hot core with orange and red rims, "
     "layered cel shading with hard-edged color bands, sharp flame tips, embers and debris flying outward, no outlines, vivid colors, ")
CAM = {"ang": "fixed camera", "orbit": "the camera slowly orbits around the explosion, buildings slide past with parallax"}
N = ("blurry, low quality, 3D render, photorealistic, smooth gradients, grey smoke, text, watermark, static, overall gray, deformed")
H, W = 480, 832
ref = Image.open("/tmp/ref.png").convert("RGB").resize((W, H))
log = {}
for name in ("ang", "orbit"):
    blk = [Image.open(p).convert("RGB").resize((W, H)) for p in sorted(glob.glob(f"/tmp/blk7/{name}/*.png"))]
    n0 = len(blk); n = ((n0 - 1 + 3) // 4) * 4 + 1  # 4n+1로 올림
    vid = blk + [blk[-1]] * (n - n0)
    mask = [Image.new("L", (W, H), 255)] * n
    for tag, refs in (("ref", [ref]),):
        frames = None
        for (h, w) in ((H, W), (384, 672)):  # 메모리가 모자라면 작은 크기로 다시
            try:
                torch.cuda.empty_cache()
                g = torch.Generator("cpu").manual_seed(1234)
                rs = None if refs is None else [r.resize((w, h)) for r in refs]
                frames = pipe(video=[v.resize((w, h)) for v in vid], mask=[m.resize((w, h)) for m in mask], reference_images=rs,
                              prompt=P + CAM[name], negative_prompt=N, height=h, width=w, num_frames=n, num_inference_steps=30,
                              guidance_scale=5.0, generator=g, output_type="pil").frames[0]
                print(name, tag, "크기", w, "x", h)
                break
            except torch.OutOfMemoryError:
                print(name, tag, w, "x", h, "메모리 부족 → 작은 크기로")
                torch.cuda.empty_cache()
        if frames is None:
            continue
        frames = frames[:n0]
        d = f"{OUT}/m7/{name}/{tag}"; os.makedirs(d, exist_ok=True)
        for i, im in enumerate(frames):
            im.save(f"{d}/f{i:03d}.png")
        a = [np.asarray(f, np.float32) for f in frames]
        mean = float(np.mean([x.mean() for x in a]))
        print(name, tag, len(frames), "장, 평균 밝기", round(mean, 1), "(0에 가까우면 검은 그림 = dtype 문제)")
        log[f"{name} {tag}"] = round(mean, 1)
    d0 = f"{OUT}/m7/{name}/blocking"; os.makedirs(d0, exist_ok=True)
    for i, b in enumerate(blk):
        b.save(f"{d0}/f{i:03d}.png")
json.dump(log, open(f"{OUT}/motion7_log.json", "w"), ensure_ascii=False, indent=1)


In [ ]:
# 4) 나란히 영상: 위 줄 ang, 아래 줄 orbit / 칸 = 블로킹 | VACE 참조 있음. 24fps, 3번 반복
import cv2
rows = {n: {k: sorted(glob.glob(f"{OUT}/m7/{n}/{k}/*.png")) for k in ("blocking", "ref")} for n in ("ang", "orbit")}
def rd(p, label):
    im = cv2.resize(cv2.imread(p), (448, 258)); cv2.putText(im, label, (6, 20), 0, 0.55, (255, 255, 255), 2); return im
T = max(len(rows[n]["blocking"]) for n in rows)
vw = cv2.VideoWriter(f"{OUT}/motion7.mp4", cv2.VideoWriter_fourcc(*"mp4v"), 24, (896, 516))
for _ in range(3):
    for t in range(T):
        vw.write(np.vstack([np.hstack([rd(v[t % len(v)], f"{n} {k}") for k, v in rows[n].items() if v]) for n in rows]))
vw.release()
!cd {OUT} && zip -q -r motion7_result.zip m7 motion7.mp4 motion7_log.json && ls -la {OUT}


In [ ]:
# 5) Colab이면 결과 zip 내려받기(Kaggle은 Output 탭)
if COLAB and OUT.startswith("/content"):
    from google.colab import files
    files.download(f"{OUT}/motion7_result.zip")
